In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
df=pd.read_csv('titanic_data_updated.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,no,second,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,yes,first,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,no,third,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,yes,first,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


spliting data

In [ ]:
df.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)
#family_size creation
df['FamilySize']=df['SibSp']+df['Parch']+1
#feature and target extract
x=df.drop(['Survived'],axis=1)
y=df['Survived']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

imputation of data

In [ ]:
imputer_transformer=ColumnTransformer(
    transformers=[
        ('age',SimpleImputer(missing_values=np.nan,strategy='mean'),['Age']),
        ('embarked',SimpleImputer(missing_values=np.nan,strategy='most_frequent'),['Embarked']),
        ('cabin',SimpleImputer(missing_values=np.nan,strategy='constant',fill_value='Missing',add_indicator=True),['Cabin'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)
imputer_transformer.set_output(transform='pandas')
imputer_transformer.fit(x_train)
x_train=imputer_transformer.transform(x_train)
x_test=imputer_transformer.transform(x_test)


In [ ]:
x_train.isnull().sum()

,0
Age,0
Embarked,0
Cabin,0
missingindicator_Cabin,0
Pclass,0
Sex,0
SibSp,0
Parch,0
Fare,0
FamilySize,0


In [ ]:
x_test.isnull().sum()

,0
Age,0
Embarked,0
Cabin,0
missingindicator_Cabin,0
Pclass,0
Sex,0
SibSp,0
Parch,0
Fare,0
FamilySize,0


outlier handling of age

In [ ]:
mean_of_age=x_train['Age'].mean()
std_of_age=x_train['Age'].std()
x_train['zscore_Age']=(x_train['Age']-mean_of_age)/std_of_age
outliers=x_train[abs(x_train['zscore_Age'])<=3]
x_train.drop(['zscore_Age'],axis=1,inplace=True)

outlier handling of fare

In [ ]:
fare_Q1=x_train['Fare'].quantile(0.25)
fare_Q3=x_train['Fare'].quantile(0.75)
fare_IQR=fare_Q3-fare_Q1
fare_minimum=max(0,fare_Q1-1.5*fare_IQR)#cap krsi
fare_maximum=fare_Q3+1.5*fare_IQR
x_train['Fare']=x_train['Fare'].clip(fare_minimum,fare_maximum)

encoding and scaling

In [ ]:
x_train['Cabin_Deck']=x_train['Cabin'].astype(str).str[0]
x_test['Cabin_Deck']=x_test['Cabin'].astype(str).str[0]

In [ ]:
encoder_scaler=ColumnTransformer(
    transformers=[
        ('pclass',OrdinalEncoder(categories=[['third','second','first']]),['Pclass']),
        ('embarked_sex_cabin_deck',OneHotEncoder(sparse_output=False,drop='first'),['Embarked','Sex','Cabin_Deck']),
        ('age_scaler',StandardScaler(),['Age']),
        ('fare_scaler',MinMaxScaler(),['Fare','FamilySize'])
    ],
    verbose_feature_names_out=False
)
encoder_scaler.set_output(transform='pandas')
encoder_scaler.fit(x_train)
x_train=encoder_scaler.transform(x_train)
x_test=encoder_scaler.transform(x_test)

In [ ]:
x_train

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_Deck_B,Cabin_Deck_C,Cabin_Deck_D,Cabin_Deck_E,Cabin_Deck_F,Cabin_Deck_G,Cabin_Deck_M,Cabin_Deck_T,Age,Fare,FamilySize
331,2.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.232263e+00,0.442804,0.0
733,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-5.004820e-01,0.201981,0.0
382,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.926161e-01,0.123131,0.0
704,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-2.694493e-01,0.122031,0.1
813,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.809667e+00,0.485920,0.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-6.545038e-01,0.118858,0.0
270,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.735977e-16,0.481647,0.0
860,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,8.857142e-01,0.219201,0.2
435,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.193580e+00,1.000000,0.3
